# Evaluasi FEM 3 Kelas dengan FER2013

Notebook ini dipakai untuk:
1. mengecek environment Conda dan package yang sudah terpasang,
2. menjalankan evaluasi model FEM `dima806/facial_emotions_image_detection`,
3. melakukan relabel FER2013 dari 7 kelas menjadi 3 kelas: **negative, neutral, positive**,
4. menghitung metrik evaluasi seperti **accuracy, precision, recall, F1-score**, dan **confusion matrix**.

> Catatan: notebook ini **tidak** mengevaluasi video, tetapi mengevaluasi **dataset gambar FER2013**.


## 1. Cek Conda dan Package yang Terpasang

Jalankan sel di bawah ini dulu untuk melihat:
- apakah `conda` tersedia,
- environment aktif apa yang sedang dipakai,
- package penting apa saja yang sudah ter-install.


In [1]:

import sys, os, subprocess, shutil, platform

print("Python executable :", sys.executable)
print("Python version    :", sys.version)
print("Platform          :", platform.platform())
print("Conda available   :", shutil.which("conda"))

if shutil.which("conda"):
    print("\n=== conda info --envs ===")
    subprocess.run(["conda", "info", "--envs"], check=False)

    print("\n=== conda list (filtered) ===")
    result = subprocess.run(["conda", "list"], capture_output=True, text=True, check=False)
    wanted = ["python", "pip", "numpy", "pandas", "scikit-learn", "opencv", "pillow", "torch", "torchvision", "transformers", "datasets"]
    if result.stdout:
        lines = result.stdout.splitlines()
        filtered = []
        for line in lines:
            low = line.lower()
            if any(w in low for w in wanted):
                filtered.append(line)
        print("\n".join(filtered[:200]) if filtered else "Tidak ada package target yang terdeteksi dari conda list.")
else:
    print("Conda tidak terdeteksi di PATH notebook ini.")


Python executable : /home/fadawkas/miniconda3/envs/vagmi/bin/python
Python version    : 3.11.15 (main, Mar 11 2026, 17:20:07) [GCC 14.3.0]
Platform          : Linux-6.17.1-300.fc43.x86_64-x86_64-with-glibc2.42
Conda available   : /home/fadawkas/miniconda3/condabin/conda

=== conda info --envs ===

# conda environments:
#
# * -> active
# + -> frozen
base                     /home/fadawkas/miniconda3
vagmi                *   /home/fadawkas/miniconda3/envs/vagmi


=== conda list (filtered) ===
datasets                     4.7.0            pypi_0           pypi
ipython                      9.10.0           pypi_0           pypi
ipython-pygments-lexers      1.1.1            pypi_0           pypi
numpy                        2.4.3            pypi_0           pypi
opencv-python                4.13.0.92        pypi_0           pypi
pandas                       3.0.1            pypi_0           pypi
pillow                       12.1.1           pypi_0           pypi
pip                         

Kalau kamu hanya ingin cek manual dari terminal Conda, pakai perintah ini:

```bash
conda info --envs
conda activate NAMA_ENV
conda list
python -V
pip list
```


## 2. Install Dependensi Jika Belum Ada

Kalau ada package yang belum terpasang, jalankan sel berikut.


In [ ]:
# Uncomment jika perlu
# !pip install -q transformers datasets scikit-learn pillow pandas torch torchvision opencv-python matplotlib

## 3. Import Library

In [2]:

from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 4. Konfigurasi Mapping Label 7 Kelas ke 3 Kelas

FER2013 asli:
- 0 = angry
- 1 = disgust
- 2 = fear
- 3 = happy
- 4 = sad
- 5 = surprise
- 6 = neutral

Di bawah ini saya pakai mapping yang konsisten dengan kode FEM kamu saat ini:
- `happy -> positive`
- `neutral -> neutral`
- lainnya -> `negative`


In [3]:

FER_LABEL_MAP = {
    0: "angry",
    1: "disgust",
    2: "fear",
    3: "happy",
    4: "sad",
    5: "surprise",
    6: "neutral"
}

MAP_7_TO_3 = {
    "happy": "positive",
    "neutral": "neutral",
    "angry": "negative",
    "disgust": "negative",
    "fear": "negative",
    "sad": "negative",
    "surprise": "negative"
}

LABELS_3 = ["negative", "neutral", "positive"]


## 5. Pilih Sumber Dataset FER2013

Ada dua opsi:

### Opsi A — dari Hugging Face
Paling mudah bila internet notebook aktif.

### Opsi B — dari file CSV Kaggle lokal
Biasanya formatnya punya kolom:
- `emotion`
- `pixels`
- `Usage` atau file train/test terpisah

Notebook ini saya siapkan untuk **Opsi B** karena paling umum dipakai di skripsi.


In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files:", path)

100%|██████████| 60.3M/60.3M [00:09<00:00, 6.45MB/s]

Extracting files...


Path to dataset files: /home/fadawkas/.cache/kagglehub/datasets/msambare/fer2013/versions/1


## 6. Load dan Split Dataset FER2013

In [ ]:

FER2013_CSV = "/home/fadawkas/.cache/kagglehub/datasets/msambare/fer2013/versions/1"
df = pd.read_csv(FER2013_CSV_PATH)
print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Jika ada kolom Usage seperti dataset Kaggle asli
if "Usage" in df.columns:
    test_df = df[df["Usage"].isin(["PublicTest", "PrivateTest"])].copy()
else:
    # fallback: pakai seluruh data sebagai set evaluasi
    test_df = df.copy()

print("\nTest/eval shape:", test_df.shape)
test_df["emotion_name"] = test_df["emotion"].map(FER_LABEL_MAP)
test_df["label_3"] = test_df["emotion_name"].map(MAP_7_TO_3)

print("\nDistribusi label 3 kelas:")
print(test_df["label_3"].value_counts())


## 7. Ubah String Pixel FER2013 menjadi Gambar

In [ ]:

def fer_pixels_to_pil(pixel_string: str) -> Image.Image:
    pixels = np.array([int(p) for p in pixel_string.split()], dtype=np.uint8)
    img = pixels.reshape(48, 48)
    return Image.fromarray(img).convert("RGB")

# contoh visualisasi
sample_img = fer_pixels_to_pil(test_df.iloc[0]["pixels"])
plt.imshow(sample_img)
plt.axis("off")
plt.title(f'GT: {test_df.iloc[0]["emotion_name"]} -> {test_df.iloc[0]["label_3"]}')
plt.show()


## 8. Load Model FEM

In [ ]:

clf = pipeline(
    "image-classification",
    model="dima806/facial_emotions_image_detection",
    device=-1  # ganti 0 jika pakai GPU
)

# test cepat
pred_example = clf(sample_img)
pred_example[:3]


## 9. Jalankan Evaluasi

> Untuk uji awal, set `MAX_SAMPLES` kecil dulu, misalnya 100 atau 500.
> Setelah yakin lancar, ubah ke seluruh data evaluasi.


In [ ]:

MAX_SAMPLES = min(len(test_df), 500)  # ubah ke len(test_df) untuk full evaluation

y_true = []
y_pred = []
rows = []

for i in range(MAX_SAMPLES):
    row = test_df.iloc[i]

    img = fer_pixels_to_pil(row["pixels"])
    gt_7 = row["emotion_name"]
    gt_3 = row["label_3"]

    pred = clf(img)[0]
    pred_7 = pred["label"].lower()
    pred_score = float(pred["score"])
    pred_3 = MAP_7_TO_3.get(pred_7, "negative")

    y_true.append(gt_3)
    y_pred.append(pred_3)

    rows.append({
        "index": i,
        "gt_7": gt_7,
        "gt_3": gt_3,
        "pred_7": pred_7,
        "pred_3": pred_3,
        "score": pred_score
    })

result_df = pd.DataFrame(rows)
result_df.head()


## 10. Hitung Metrik Evaluasi

In [ ]:

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("=== HASIL EVALUASI FEM 3 KELAS ===")
print(f"Jumlah sampel       : {len(y_true)}")
print(f"Accuracy            : {acc:.4f}")
print(f"Precision Macro     : {precision_macro:.4f}")
print(f"Recall Macro        : {recall_macro:.4f}")
print(f"F1-score Macro      : {f1_macro:.4f}")
print(f"Precision Weighted  : {precision_weighted:.4f}")
print(f"Recall Weighted     : {recall_weighted:.4f}")
print(f"F1-score Weighted   : {f1_weighted:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, labels=LABELS_3, zero_division=0))


## 11. Confusion Matrix

In [ ]:

cm = confusion_matrix(y_true, y_pred, labels=LABELS_3)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS_3)
disp.plot()
plt.title("Confusion Matrix FEM 3 Kelas")
plt.show()

cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS_3], columns=[f"pred_{x}" for x in LABELS_3])
cm_df


## 12. Simpan Hasil Evaluasi

In [ ]:

result_df.to_csv("hasil_prediksi_fem_3kelas.csv", index=False)

summary_df = pd.DataFrame([{
    "samples": len(y_true),
    "accuracy": acc,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "precision_weighted": precision_weighted,
    "recall_weighted": recall_weighted,
    "f1_weighted": f1_weighted
}])

summary_df.to_csv("ringkasan_evaluasi_fem_3kelas.csv", index=False)

print("Saved:")
print("- hasil_prediksi_fem_3kelas.csv")
print("- ringkasan_evaluasi_fem_3kelas.csv")


## 13. Analisis Error per Kelas

Bagian ini membantu melihat kelas mana yang paling sering tertukar.


In [ ]:

misclassified = result_df[result_df["gt_3"] != result_df["pred_3"]].copy()
print("Jumlah salah klasifikasi:", len(misclassified))
misclassified.head(20)


## 14. Catatan untuk Penulisan Skripsi

Kalimat yang bisa dipakai:

> Evaluasi model Facial Emotion Model (FEM) dilakukan menggunakan dataset FER2013. Label asli dataset yang terdiri atas tujuh kelas emosi, yaitu angry, disgust, fear, happy, sad, surprise, dan neutral, kemudian dipetakan ulang menjadi tiga kelas, yaitu negative, neutral, dan positive. Selanjutnya, prediksi model dibandingkan dengan label hasil pemetaan untuk menghitung metrik accuracy, precision, recall, F1-score, serta confusion matrix.

Kalau nanti kamu ingin, mapping `surprise` bisa diubah menjadi `positive`, tetapi harus dijelaskan konsisten di metodologi.
